# Catálogo de dados

Este notebook documenta as tabelas do projeto Olist nas camadas
Bronze, Silver e Gold do catálogo `olist_mvp`.

A documentação apresenta o significado das tabelas e de seus
campos, os tipos de dados, as chaves, os domínios esperados
e a origem das informações.

## Organização das camadas

- **Bronze:** dados dos oito arquivos CSV, com os campos de origem
  armazenados como texto e metadados de ingestão.
- **Silver:** dados com tipos definidos, padronização de campos
  e identificação de falhas de conversão. Os registros são
  preservados, inclusive quando apresentam problemas de qualidade.
- **Gold:** indicadores agregados de vendas mensais, vendas por
  categoria e avaliações por pontualidade da entrega.

As chaves identificadas nas verificações de qualidade descrevem
a estrutura observada nos dados. Essas verificações não equivalem
à criação de restrições de chave primária ou estrangeira no banco.

Os domínios esperados representam regras de interpretação e
qualidade. Valores observados fora desses domínios são documentados
e não são automaticamente excluídos.

In [0]:
tabelas_origem = [
    "customers", "orders", "order_items", "payments",
    "reviews", "products", "sellers", "category_translation"
]

tabelas_por_camada = {
    "bronze": tabelas_origem,
    "silver": tabelas_origem,
    "gold": [
        "vendas_mensais",
        "vendas_por_categoria",
        "avaliacoes_por_atraso"
    ]
}

estrutura = []

for camada, tabelas in tabelas_por_camada.items():
    for tabela in tabelas:
        nome_completo = f"olist_mvp.{camada}.{tabela}"

        for posicao, campo in enumerate(
            spark.table(nome_completo).schema.fields,
            start=1
        ):
            estrutura.append((
                camada,
                tabela,
                posicao,
                campo.name,
                campo.dataType.simpleString()
            ))

estrutura_catalogo = spark.createDataFrame(
    estrutura,
    """
    camada string,
    tabela string,
    posicao int,
    coluna string,
    tipo_dado string
    """
)

display(
    estrutura_catalogo.orderBy("camada", "tabela", "posicao")
)

camada,tabela,posicao,coluna,tipo_dado
bronze,category_translation,1,product_category_name,string
bronze,category_translation,2,product_category_name_english,string
bronze,category_translation,3,_source_file,string
bronze,category_translation,4,_ingestion_timestamp,timestamp
bronze,customers,1,customer_id,string
bronze,customers,2,customer_unique_id,string
bronze,customers,3,customer_zip_code_prefix,string
bronze,customers,4,customer_city,string
bronze,customers,5,customer_state,string
bronze,customers,6,_source_file,string


## Estrutura e identificação dos registros

As camadas Bronze e Silver possuem oito tabelas cada. A Silver
preserva a quantidade de registros da Bronze, aplicando as
conversões de tipos e padronizações documentadas no tratamento.

As chaves abaixo foram verificadas nos dados. Não foram criadas
restrições de chave primária ou estrangeira no banco.

| Tabela na Bronze e Silver | O que cada linha representa | Chave identificada |
|---|---|---|
| customers | Cadastro de cliente associado a um pedido | customer_id |
| orders | Um pedido | order_id |
| order_items | Um item de um pedido | order_id + order_item_id |
| payments | Um registro de pagamento de um pedido | order_id + payment_sequential |
| reviews | Uma avaliação associada a um pedido | review_id + order_id |
| products | Um produto | product_id |
| sellers | Um vendedor | seller_id |
| category_translation | Tradução de uma categoria de produto | product_category_name |

### Particularidades das chaves

O campo `customer_unique_id` permite identificar um mesmo cliente
em diferentes compras. Já `customer_id` identifica o cadastro
associado ao pedido e é utilizado na ligação com `orders`.

Um pedido pode possuir vários itens, pagamentos e avaliações.
Essas relações exigem cuidado nas associações para evitar
multiplicação de registros e valores.

Em `reviews`, `review_id` apresenta repetições. A combinação
`review_id + order_id` não apresentou valores ausentes nem
repetições na base analisada.

### Tabelas analíticas da Gold

| Tabela | O que cada linha representa | Campo identificador |
|---|---|---|
| vendas_mensais | Indicadores dos pedidos entregues agrupados pelo mês da compra | mes_compra |
| vendas_por_categoria | Indicadores dos itens de pedidos entregues agrupados por categoria | categoria |
| avaliacoes_por_atraso | Indicadores de entregas e avaliações agrupados pela situação de pontualidade | situacao_entrega |

Na análise mensal, os itens são agregados por pedido antes da
associação com os pedidos. Na análise de pontualidade, as avaliações
válidas são agregadas por pedido antes da associação com as entregas.

Na análise por categoria, um pedido pode participar de mais de uma
categoria. Portanto, a soma das contagens de pedidos das categorias
não representa o total de pedidos distintos.

## Dicionário dos campos — Bronze e Silver

As descrições abaixo se aplicam às tabelas Bronze e Silver.
Todos os campos de origem são armazenados como `string` na Bronze.
Os tipos apresentados nas tabelas abaixo correspondem à Silver.

Os domínios representam valores esperados para interpretação e
verificação de qualidade, não restrições impostas ao banco.
Valores ausentes ou inconsistentes permanecem documentados.
Limites superiores de negócio não definidos não são substituídos
pelos maiores valores observados na amostra.

### Clientes — customers

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| customer_id | string | Identificador do cadastro associado ao pedido; preenchido e único na tabela. |
| customer_unique_id | string | Identificador do cliente ao longo das compras; pode se repetir. |
| customer_zip_code_prefix | string | Prefixo do CEP; de um a cinco dígitos no padrão verificado. Mantido como texto. |
| customer_city | string | Nome da cidade do cliente; texto, sem lista fechada definida no projeto. |
| customer_state | string | Sigla da unidade federativa do cliente; uma das 27 UFs brasileiras. |

### Pedidos — orders

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| order_id | string | Identificador do pedido; preenchido e único na tabela. |
| customer_id | string | Referência ao cadastro em customers.customer_id. |
| order_status | string | Situação do pedido: created, approved, invoiced, processing, shipped, delivered, unavailable ou canceled. |
| order_purchase_timestamp | timestamp_ntz | Data e hora da compra; referência temporal inicial do pedido. |
| order_approved_at | timestamp_ntz | Data e hora da aprovação; quando preenchida, espera-se que não anteceda a compra. |
| order_delivered_carrier_date | timestamp_ntz | Data e hora da entrega à transportadora; espera-se que não anteceda a compra. |
| order_delivered_customer_date | timestamp_ntz | Data e hora da entrega ao cliente; espera-se que não anteceda o envio. Necessária para confirmar a data de entrega de pedidos delivered. |
| order_estimated_delivery_date | timestamp_ntz | Data prevista para entrega; espera-se que não anteceda a compra. Utilizada como prazo na Gold. |

A ausência de datas pode depender da situação do pedido. Por exemplo,
um pedido cancelado pode não ter data de entrega ao cliente.
As datas devem ser interpretadas em conjunto com o status.

### Itens dos pedidos — order_items

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| order_id | string | Referência ao pedido em orders.order_id. |
| order_item_id | int | Sequência do item dentro do pedido; inteiro maior ou igual a 1. Compõe a chave com order_id. |
| product_id | string | Referência ao produto em products.product_id. |
| seller_id | string | Referência ao vendedor em sellers.seller_id. |
| shipping_limit_date | timestamp_ntz | Data e hora limite de envio do item; deve representar uma data válida. |
| price | decimal(18,2) | Preço do item em reais, sem frete; valor esperado maior que zero. |
| freight_value | decimal(18,2) | Valor do frete atribuído ao item, em reais; valor esperado maior ou igual a zero. |

### Pagamentos — payments

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| order_id | string | Referência ao pedido em orders.order_id. |
| payment_sequential | int | Sequência do registro de pagamento dentro do pedido; inteiro maior ou igual a 1. Compõe a chave com order_id. |
| payment_type | string | Forma de pagamento: credit_card, boleto, voucher, debit_card ou not_defined. O último valor indica informação não definida. |
| payment_installments | int | Quantidade de parcelas; valor esperado maior ou igual a 1. Registros com zero foram sinalizados e preservados. |
| payment_value | decimal(18,2) | Valor do registro de pagamento, em reais; esperado não negativo. Valores iguais a zero foram analisados separadamente e preservados. |

### Avaliações — reviews

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| review_id | string | Identificador da avaliação; pode se repetir isoladamente. Compõe a chave observada com order_id. |
| order_id | string | Referência ao pedido avaliado em orders.order_id. |
| review_score | int | Nota da avaliação; inteiro de 1 a 5. |
| review_comment_title | string | Título do comentário; texto livre, podendo estar ausente. |
| review_comment_message | string | Conteúdo do comentário; texto livre, podendo estar ausente. |
| review_creation_date | timestamp_ntz | Data de criação do registro de avaliação; deve representar uma data válida. |
| review_answer_timestamp | timestamp_ntz | Data e hora da resposta à avaliação; espera-se que não anteceda review_creation_date. |

A ausência de comentário não significa ausência de nota.
Na Gold, as notas válidas são agregadas por pedido antes da
comparação entre situações de entrega.

### Produtos — products

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| product_id | string | Identificador do produto; preenchido e único na tabela. |
| product_category_name | string | Nome original da categoria; pode estar ausente. A correspondência com category_translation foi verificada. |
| product_name_lenght | int | Comprimento do nome do produto em caracteres; inteiro não negativo. |
| product_description_lenght | int | Comprimento da descrição do produto em caracteres; inteiro não negativo. |
| product_photos_qty | int | Quantidade de fotos do produto; inteiro não negativo. |
| product_weight_g | decimal(18,3) | Peso do produto em gramas; valor esperado maior que zero. |
| product_length_cm | decimal(18,3) | Comprimento do produto em centímetros; valor esperado maior que zero. |
| product_height_cm | decimal(18,3) | Altura do produto em centímetros; valor esperado maior que zero. |
| product_width_cm | decimal(18,3) | Largura do produto em centímetros; valor esperado maior que zero. |

A grafia original `lenght` foi preservada nos nomes dos campos.
As categorias permanecem com os nomes de origem; a Gold agrupa
categorias ausentes como “Não informada”.

### Vendedores — sellers

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| seller_id | string | Identificador do vendedor; preenchido e único na tabela. |
| seller_zip_code_prefix | string | Prefixo do CEP; de um a cinco dígitos no padrão verificado. Mantido como texto. |
| seller_city | string | Nome da cidade do vendedor; texto, sem lista fechada definida no projeto. |
| seller_state | string | Sigla da unidade federativa do vendedor; uma das 27 UFs brasileiras. |

As UFs consideradas são: AC, AL, AP, AM, BA, CE, DF, ES, GO,
MA, MT, MS, MG, PA, PB, PR, PE, PI, RJ, RN, RS, RO, RR,
SC, SP, SE e TO.

A verificação do formato do prefixo do CEP não comprova a existência
do endereço nem sua correspondência com a cidade e a UF.

### Tradução de categorias — category_translation

| Campo | Tipo na Silver | Descrição e domínio esperado |
|---|---|---|
| product_category_name | string | Nome original da categoria; preenchido e único nesta tabela. |
| product_category_name_english | string | Tradução do nome da categoria para o inglês. |

Esta tabela é mantida como referência. As análises da Gold utilizam
os nomes originais das categorias, sem depender da tradução.

### Metadados de processamento

| Campo | Camadas | Tipo | Descrição e domínio esperado |
|---|---|---|---|
| _source_file | Bronze e Silver | string | Nome do arquivo CSV de origem; um dos oito arquivos utilizados no projeto. |
| _ingestion_timestamp | Bronze e Silver | timestamp | Instante da ingestão na Bronze; preservado na Silver. |
| _colunas_com_falha_conversao | Silver | array<string> | Lista dos campos cujo valor preenchido não pôde ser convertido após a normalização; lista vazia quando não há falhas. |
| _silver_timestamp | Silver | timestamp | Instante do processamento da tabela na Silver. |

Os campos `timestamp_ntz` preservam datas e horários sem informação
de fuso horário. Os metadados de processamento utilizam `timestamp`
e sua exibição depende do fuso configurado na sessão.

## Dicionário dos indicadores — Gold

As três tabelas utilizam pedidos com status `delivered`.
Os valores monetários são expressos em reais e não representam
receita líquida ou lucro da Olist.

### Vendas mensais — vendas_mensais

Origem: `silver.orders` e `silver.order_items`.

Os itens são agregados por `order_id`, calculando a soma de `price`
e a quantidade de itens. O resultado é associado aos pedidos
entregues por uma junção à esquerda, preservando pedidos sem
correspondência em itens.

Pedidos sem data de compra são excluídos desta análise.
O agrupamento utiliza o mês de `order_purchase_timestamp`.

| Campo | Tipo | Descrição e cálculo | Domínio esperado |
|---|---|---|---|
| mes_compra | date | Primeiro dia do mês da compra, utilizado como referência mensal. | Data preenchida, com dia igual a 1; única na tabela. |
| quantidade_pedidos | bigint | Contagem dos pedidos entregues com data de compra no mês. | Inteiro maior que zero para os meses presentes na tabela. |
| pedidos_com_itens | bigint | Contagem dos pedidos cuja soma de preços dos itens não é nula. | Entre zero e quantidade_pedidos. |
| quantidade_itens | bigint | Soma das quantidades de itens dos pedidos do mês. | Inteiro positivo quando existem itens; pode ser nulo se não houver correspondências. |
| valor_total_itens | decimal(38,2) | Soma dos preços dos itens dos pedidos do mês, sem frete. | Esperado positivo quando existem preços válidos; pode ser nulo na ausência deles. |
| valor_medio_itens_por_pedido | decimal(29,2) | Média do valor dos itens por pedido, arredondada para duas casas decimais. Pedidos sem valor disponível não participam da média. | Esperado positivo quando existem preços válidos; nulo quando não há valores para calcular a média. |

O indicador `pedidos_com_itens` utiliza a disponibilidade do valor
agregado dos itens como medida de cobertura. Um pedido com itens,
mas com todos os preços nulos, não seria contado nesse indicador.

Meses sem pedidos elegíveis não aparecem na tabela. No gráfico,
esses meses são apresentados como lacunas, sem preenchimento com zero.

### Vendas por categoria — vendas_por_categoria

Origem: `silver.order_items`, `silver.orders` e `silver.products`.

Os itens são associados aos pedidos entregues por `order_id`,
utilizando uma junção interna. Em seguida, uma junção à esquerda
com os produtos por `product_id` recupera a categoria.

Categorias ausentes são substituídas por “Não informada”.
Os indicadores são agrupados por categoria.

| Campo | Tipo | Descrição e cálculo | Domínio esperado |
|---|---|---|---|
| categoria | string | Nome original da categoria ou “Não informada” quando ausente. | Texto preenchido e único na tabela. |
| quantidade_pedidos | bigint | Contagem distinta de order_id na categoria. | Inteiro maior que zero. |
| quantidade_itens | bigint | Contagem dos registros de itens na categoria. | Inteiro maior ou igual a quantidade_pedidos. |
| valor_total_itens | decimal(28,2) | Soma de price dos itens da categoria, sem frete. | Esperado positivo quando existem preços válidos; pode ser nulo se todos os preços estiverem ausentes. |
| participacao_percentual | decimal(35,2) | 100 × valor_total_itens da categoria ÷ soma do valor_total_itens de todas as categorias; arredondamento para duas casas decimais. | Entre 0 e 100 quando os valores são não negativos e o total é positivo; nulo se o cálculo não tiver valores válidos ou se o total não for positivo. |

O denominador da participação inclui “Não informada”.
A soma dos percentuais exibidos pode diferir ligeiramente de 100%
por causa do arredondamento.

A contagem de pedidos não é aditiva entre categorias, pois um
mesmo pedido pode conter produtos de categorias diferentes.

### Avaliações por atraso — avaliacoes_por_atraso

Origem: `silver.orders` e `silver.reviews`.

São selecionados pedidos entregues com datas de compra, envio,
entrega e previsão preenchidas, respeitando os seguintes critérios:

- Envio igual ou posterior à compra.
- Entrega igual ou posterior ao envio.
- Previsão de entrega igual ou posterior à compra.

Calcula-se a diferença, em dias de calendário, entre a entrega
ao cliente e a previsão. Diferenças positivas representam atraso;
diferenças iguais a zero ou negativas representam entrega no prazo.

As avaliações com notas entre 1 e 5 são agrupadas por `order_id`,
calculando uma nota média por pedido. Essa média é associada aos
pedidos elegíveis por uma junção à esquerda.

| Campo | Tipo | Descrição e cálculo | Domínio esperado |
|---|---|---|---|
| situacao_entrega | string | Classificação do pedido conforme a diferença entre entrega e previsão. | “No prazo” ou “Com atraso”; única na tabela. |
| quantidade_pedidos | bigint | Contagem dos pedidos elegíveis em cada situação de entrega. | Inteiro maior que zero para os grupos presentes na tabela. |
| pedidos_com_avaliacao | bigint | Contagem dos pedidos com pelo menos uma avaliação válida. | Entre zero e quantidade_pedidos. |
| nota_media | double | Média das notas médias dos pedidos do grupo, arredondada para duas casas decimais. | Entre 1 e 5; nula quando não existem avaliações válidas no grupo. |
| media_dias_atraso | double | Média dos dias de atraso dos pedidos atrasados, arredondada para duas casas decimais. | Maior ou igual a 1 para “Com atraso”; nula para “No prazo”, pois não se aplica. |

Cada pedido com avaliação válida possui o mesmo peso no cálculo
de `nota_media`, independentemente da quantidade de avaliações.

A média de dias de atraso considera todos os pedidos atrasados
elegíveis, inclusive aqueles sem avaliação. Portanto, seu conjunto
de pedidos pode diferir daquele utilizado na média das notas.

Os campos derivados por pedido, como a nota média individual e
a diferença de dias, são intermediários do processamento e não
são armazenados como colunas dessas tabelas agregadas.

## Origem dos dados e transformações

A linhagem descreve o caminho dos dados desde os arquivos de
origem até os indicadores analíticos.

### Dos arquivos para a Bronze

Os oito arquivos CSV são lidos do volume
`/Volumes/olist_mvp/bronze/raw_files`.

| Arquivo de origem | Tabela na Bronze |
|---|---|
| olist_customers_dataset.csv | olist_mvp.bronze.customers |
| olist_orders_dataset.csv | olist_mvp.bronze.orders |
| olist_order_items_dataset.csv | olist_mvp.bronze.order_items |
| olist_order_payments_dataset.csv | olist_mvp.bronze.payments |
| olist_order_reviews_dataset.csv | olist_mvp.bronze.reviews |
| olist_products_dataset.csv | olist_mvp.bronze.products |
| olist_sellers_dataset.csv | olist_mvp.bronze.sellers |
| product_category_name_translation.csv | olist_mvp.bronze.category_translation |

A leitura utiliza cabeçalho, codificação UTF-8 e suporte a campos
com múltiplas linhas. A inferência automática de tipos é desativada,
mantendo os campos de origem como texto.

São acrescentados o nome do arquivo de origem e o instante da
ingestão. As tabelas são salvas em formato Delta, com substituição
do conteúdo anterior a cada execução.

O arquivo de geolocalização não integra o escopo deste projeto.

### Da Bronze para a Silver

Cada tabela Silver utiliza a tabela Bronze de mesmo nome.
Não são realizadas junções entre tabelas nesta etapa.

As transformações aplicadas são:

- Remoção de espaços no início e no fim dos campos textuais
  estruturados, preservando os metadados de origem.
- Conversão de textos vazios ou compostos apenas por espaços
  em valores nulos.
- Preservação do conteúdo dos títulos e comentários de avaliações
  quando preenchidos.
- Conversão de datas, números inteiros e valores decimais conforme
  os tipos apresentados no dicionário.
- Padronização das siglas de UF em letras maiúsculas.
- Registro dos campos com falha de conversão em
  `_colunas_com_falha_conversao`.
- Inclusão do instante de processamento em `_silver_timestamp`.

Uma falha de conversão produz um valor nulo no campo convertido
e registra seu nome no metadado de falhas. O valor de origem
permanece disponível na Bronze.

Não são removidos registros por ausência de campos, inconsistências
cronológicas, valores fora do domínio ou valores extremos.
As verificações de qualidade identificam essas situações e
fundamentam os critérios de uso na Gold.

A quantidade de linhas da Silver é comparada com a Bronze
durante o processamento. As tabelas são salvas em formato Delta,
substituindo o conteúdo anterior.

### Da Silver para a Gold

| Tabela Gold | Tabelas Silver utilizadas | Transformação principal |
|---|---|---|
| vendas_mensais | orders e order_items | Seleção de pedidos entregues, agregação dos itens por pedido, associação e agrupamento pelo mês da compra. |
| vendas_por_categoria | orders, order_items e products | Seleção dos itens de pedidos entregues, associação aos produtos, agrupamento por categoria e cálculo da participação no valor total. |
| avaliacoes_por_atraso | orders e reviews | Seleção de entregas com datas consistentes, classificação da pontualidade, agregação das avaliações por pedido e cálculo dos indicadores por situação de entrega. |

As tabelas `customers`, `payments`, `sellers` e
`category_translation` são tratadas e documentadas, mas não
participam diretamente dos três indicadores atuais da Gold.

Os filtros da Gold afetam apenas os recortes analíticos.
Os registros excluídos desses recortes permanecem disponíveis
nas camadas anteriores.

### Ordem de execução

1. `01_bronze_ingestão`: leitura dos arquivos e diagnóstico inicial.
2. `02_silver_tratamento`: conversões, padronizações e verificações.
3. `03_gold_análise`: construção dos indicadores e interpretação.
4. `04_catálogo_dados`: consulta da estrutura e documentação.

Quando os dados ou as regras forem alterados, as camadas
dependentes devem ser atualizadas nessa ordem.

In [0]:
descricoes_origem = {
    "customers": (
        "Cadastros de clientes associados aos pedidos. "
        "Chave observada: customer_id. "
        "customer_unique_id identifica o cliente entre compras."
    ),
    "orders": (
        "Pedidos, com status e datas de compra, aprovação, envio, "
        "entrega e previsão. Chave observada: order_id."
    ),
    "order_items": (
        "Itens dos pedidos, com produto, vendedor, preço e frete. "
        "Chave observada: order_id + order_item_id."
    ),
    "payments": (
        "Registros de pagamento dos pedidos, com forma, parcelas "
        "e valor. Chave observada: order_id + payment_sequential."
    ),
    "reviews": (
        "Avaliações dos pedidos, com notas, comentários e datas. "
        "Chave observada: review_id + order_id. "
        "review_id não é único isoladamente."
    ),
    "products": (
        "Produtos, com categoria e características do cadastro "
        "e das dimensões físicas. Chave observada: product_id."
    ),
    "sellers": (
        "Vendedores e sua localização por prefixo de CEP, cidade "
        "e UF. Chave observada: seller_id."
    ),
    "category_translation": (
        "Tradução dos nomes originais das categorias para inglês. "
        "Chave observada: product_category_name. "
        "A Gold utiliza os nomes originais."
    )
}

descricoes_camadas = {
    "bronze": (
        "Camada Bronze: campos de origem armazenados como texto, "
        "com nome do arquivo e instante da ingestão. "
    ),
    "silver": (
        "Camada Silver: tipos convertidos, campos padronizados "
        "e falhas de conversão identificadas. "
        "Registros preservados, inclusive com problemas de qualidade. "
    )
}

comentarios_tabelas = {
    f"olist_mvp.{camada}.{tabela}": contexto + descricao
    for camada, contexto in descricoes_camadas.items()
    for tabela, descricao in descricoes_origem.items()
}

comentarios_tabelas.update({
    "olist_mvp.gold.vendas_mensais": (
        "Indicadores mensais dos pedidos entregues, pelo mês da compra. "
        "Uma linha por mes_compra. Origem: Silver orders e order_items. "
        "Itens agregados por pedido antes da associação. "
        "Valores em reais, sem frete; não representam receita líquida."
    ),
    "olist_mvp.gold.vendas_por_categoria": (
        "Indicadores dos itens de pedidos entregues por categoria. "
        "Uma linha por categoria. Origem: Silver orders, order_items "
        "e products. Participação calculada sobre o valor dos itens "
        "de todas as categorias, incluindo Não informada. "
        "Valores sem frete. Contagens de pedidos não aditivas "
        "entre categorias."
    ),
    "olist_mvp.gold.avaliacoes_por_atraso": (
        "Indicadores de pedidos entregues por situação de pontualidade. "
        "Uma linha por situacao_entrega. Origem: Silver orders e reviews. "
        "Exige datas preenchidas e cronologicamente consistentes. "
        "Atraso medido em dias de calendário. Notas válidas de 1 a 5, "
        "agregadas primeiro por pedido. Comparação descritiva, "
        "sem demonstração de causalidade."
    )
})

for tabela, descricao in comentarios_tabelas.items():
    nome_sql = ".".join(f"`{parte}`" for parte in tabela.split("."))
    texto_sql = descricao.replace("'", "''")

    spark.sql(
        f"COMMENT ON TABLE {nome_sql} IS '{texto_sql}'"
    )

print(f"Descrições registradas em {len(comentarios_tabelas)} tabelas.")

Descrições registradas em 19 tabelas.


In [0]:
descricoes_campos = {
    "customer_id": "Identificador do cadastro do cliente associado ao pedido; ligação entre customers e orders.",
    "customer_unique_id": "Identificador do cliente ao longo das compras; pode se repetir entre cadastros.",
    "customer_zip_code_prefix": "Prefixo do CEP do cliente, mantido como texto; padrão verificado de um a cinco dígitos.",
    "customer_city": "Nome da cidade do cliente.",
    "customer_state": "Sigla da UF do cliente; domínio esperado: uma das 27 UFs brasileiras.",

    "order_id": "Identificador do pedido; chave de orders e referência nas tabelas de itens, pagamentos e avaliações.",
    "order_status": "Situação do pedido: created, approved, invoiced, processing, shipped, delivered, unavailable ou canceled.",
    "order_purchase_timestamp": "Data e hora da compra; referência temporal inicial do pedido.",
    "order_approved_at": "Data e hora da aprovação; espera-se que não anteceda a compra.",
    "order_delivered_carrier_date": "Data e hora da entrega à transportadora; espera-se que não anteceda a compra.",
    "order_delivered_customer_date": "Data e hora da entrega ao cliente; espera-se que não anteceda o envio.",
    "order_estimated_delivery_date": "Data prevista para entrega; referência de prazo utilizada na Gold.",

    "order_item_id": "Sequência do item no pedido; inteiro esperado maior ou igual a 1. Compõe a chave com order_id.",
    "product_id": "Identificador do produto; chave de products e referência em order_items.",
    "seller_id": "Identificador do vendedor; chave de sellers e referência em order_items.",
    "shipping_limit_date": "Data e hora limite para envio do item pelo vendedor.",
    "price": "Preço do item em reais, sem frete; valor esperado maior que zero.",
    "freight_value": "Frete atribuído ao item, em reais; valor esperado maior ou igual a zero.",

    "payment_sequential": "Sequência do pagamento no pedido; inteiro esperado maior ou igual a 1. Compõe a chave com order_id.",
    "payment_type": "Forma de pagamento: credit_card, boleto, voucher, debit_card ou not_defined. not_defined indica informação não definida.",
    "payment_installments": "Quantidade de parcelas; inteiro esperado maior ou igual a 1. Valores zero foram sinalizados e preservados.",
    "payment_value": "Valor do registro de pagamento em reais; esperado não negativo. Valores zero foram analisados separadamente.",

    "review_id": "Identificador da avaliação; não é único isoladamente. Compõe a chave observada com order_id.",
    "review_score": "Nota da avaliação; inteiro esperado entre 1 e 5.",
    "review_comment_title": "Título do comentário da avaliação; texto livre, podendo estar ausente.",
    "review_comment_message": "Conteúdo do comentário da avaliação; texto livre, podendo estar ausente.",
    "review_creation_date": "Data de criação do registro de avaliação.",
    "review_answer_timestamp": "Data e hora da resposta à avaliação; espera-se que não anteceda review_creation_date.",

    "product_category_name": "Nome original da categoria do produto; ligação entre products e category_translation.",
    "product_category_name_english": "Tradução do nome da categoria para o inglês.",
    "product_name_lenght": "Comprimento do nome do produto em caracteres; inteiro não negativo. Grafia original do campo preservada.",
    "product_description_lenght": "Comprimento da descrição do produto em caracteres; inteiro não negativo. Grafia original do campo preservada.",
    "product_photos_qty": "Quantidade de fotos do produto; inteiro não negativo.",
    "product_weight_g": "Peso do produto em gramas; valor esperado maior que zero.",
    "product_length_cm": "Comprimento do produto em centímetros; valor esperado maior que zero.",
    "product_height_cm": "Altura do produto em centímetros; valor esperado maior que zero.",
    "product_width_cm": "Largura do produto em centímetros; valor esperado maior que zero.",

    "seller_zip_code_prefix": "Prefixo do CEP do vendedor, mantido como texto; padrão verificado de um a cinco dígitos.",
    "seller_city": "Nome da cidade do vendedor.",
    "seller_state": "Sigla da UF do vendedor; domínio esperado: uma das 27 UFs brasileiras.",

    "_source_file": "Nome do arquivo CSV de origem; um dos oito arquivos utilizados no projeto.",
    "_ingestion_timestamp": "Instante da ingestão na Bronze; preservado na Silver.",
    "_colunas_com_falha_conversao": "Lista dos campos preenchidos que falharam na conversão após normalização; vazia quando não há falhas.",
    "_silver_timestamp": "Instante do processamento da tabela na Silver."
}

descricoes_gold = {
    "vendas_mensais": {
        "mes_compra": "Primeiro dia do mês da compra; identifica o agrupamento mensal dos pedidos entregues.",
        "quantidade_pedidos": "Contagem de pedidos entregues com data de compra no mês.",
        "pedidos_com_itens": "Contagem de pedidos cuja soma dos preços dos itens não é nula; entre zero e quantidade_pedidos.",
        "quantidade_itens": "Soma das quantidades de itens dos pedidos do mês; nula se não houver correspondências.",
        "valor_total_itens": "Soma dos preços dos itens dos pedidos do mês, em reais e sem frete.",
        "valor_medio_itens_por_pedido": "Média do valor dos itens por pedido, em reais, arredondada para duas casas; exclui pedidos sem valor disponível."
    },
    "vendas_por_categoria": {
        "categoria": "Nome original da categoria ou Não informada quando ausente; identifica o agrupamento.",
        "quantidade_pedidos": "Contagem distinta de pedidos entregues na categoria; não aditiva entre categorias.",
        "quantidade_itens": "Contagem dos itens de pedidos entregues na categoria.",
        "valor_total_itens": "Soma dos preços dos itens da categoria, em reais e sem frete.",
        "participacao_percentual": "100 vezes o valor da categoria dividido pelo valor de todas as categorias, incluindo Não informada; duas casas decimais. Esperado entre 0 e 100; nulo se o cálculo não for possível."
    },
    "avaliacoes_por_atraso": {
        "situacao_entrega": "No prazo ou Com atraso. Atraso ocorre quando o dia da entrega é posterior ao dia previsto.",
        "quantidade_pedidos": "Contagem de pedidos entregues do grupo com datas disponíveis e cronologicamente consistentes.",
        "pedidos_com_avaliacao": "Contagem de pedidos do grupo com pelo menos uma nota válida entre 1 e 5.",
        "nota_media": "Média das notas médias por pedido, com o mesmo peso para cada pedido avaliado; duas casas decimais. Entre 1 e 5 ou nula na ausência de avaliações válidas.",
        "media_dias_atraso": "Média dos dias de atraso dos pedidos atrasados, incluindo os sem avaliação; duas casas decimais. Nula para No prazo, pois não se aplica."
    }
}

comentarios_colunas = []
campos_sem_descricao = []

for camada, tabelas in tabelas_por_camada.items():
    for tabela in tabelas:
        nome_completo = f"olist_mvp.{camada}.{tabela}"
        definicoes = (
            descricoes_gold[tabela]
            if camada == "gold"
            else descricoes_campos
        )

        for campo in spark.table(nome_completo).schema.fields:
            descricao = definicoes.get(campo.name)

            if descricao is None:
                campos_sem_descricao.append(
                    f"{nome_completo}.{campo.name}"
                )
                continue

            if camada == "bronze" and not campo.name.startswith("_"):
                descricao += (
                    " Na Bronze, armazenado como texto de origem; "
                    "o domínio descrito orienta a análise de qualidade."
                )

            comentarios_colunas.append((
                nome_completo,
                campo.name,
                descricao
            ))

if campos_sem_descricao:
    raise ValueError(
        f"Campos sem descrição: {campos_sem_descricao}"
    )

for tabela, coluna, descricao in comentarios_colunas:
    nome_sql = ".".join(
        f"`{parte}`" for parte in tabela.split(".") + [coluna]
    )
    texto_sql = descricao.replace("'", "''")

    spark.sql(
        f"COMMENT ON COLUMN {nome_sql} IS '{texto_sql}'"
    )

print(
    f"Descrições registradas em {len(comentarios_colunas)} colunas."
)

Descrições registradas em 158 colunas.


## Conclusão do catálogo de dados

Foram documentadas 19 tabelas das camadas Bronze, Silver e Gold,
totalizando 158 campos, incluindo os metadados de processamento
e as ocorrências de um mesmo campo em diferentes tabelas.

A documentação reúne o significado dos campos, os tipos de dados,
as chaves observadas, os domínios esperados e as transformações
entre as camadas. As descrições também foram registradas nas
tabelas e colunas do catálogo do Databricks, permitindo consultá-las
junto à estrutura dos dados.

As regras documentadas apoiam a interpretação e a análise de
qualidade. Elas não constituem restrições automáticas no banco
nem garantem que todos os registros atendam aos domínios esperados.
As inconsistências identificadas estão apresentadas nos notebooks
Bronze e Silver.

### Manutenção da documentação

Alterações nos arquivos, nas estruturas ou nas regras de cálculo
devem ser acompanhadas da atualização deste catálogo.

Após recriar as tabelas, recomenda-se executar novamente as células
de documentação e conferir os comentários no Databricks, mantendo
as descrições alinhadas à versão atual dos dados.